In [9]:
import pandas as pd
import ustreasurycurve as ustcurve
from datetime import datetime
from zoneinfo import ZoneInfo
from io import StringIO

# Today's date in Czech time
today = pd.Timestamp(datetime.now(ZoneInfo("Europe/Prague")).date())
start = today - pd.Timedelta(days=31)

In [ ]:
# Get Treasury data
df = ustcurve.nominal_rates(
    start.strftime("%Y-%m-%d"),
    today.strftime("%Y-%m-%d")
)[['date', '2y', '10y']]

df['date'] = pd.to_datetime(df['date'])

# Business days for the last 31 days
dates = pd.DataFrame({
    'date': pd.date_range(start, today, freq='B')
})

# Match each business day to the latest available Treasury observation
date2y10y = pd.merge_asof(
    dates,
    df.assign(source_date=df['date']).sort_values('date'),
    on='date',
    direction='backward'
)

# One-observation lag
date2y10y[['2y', '10y']] = date2y10y[['2y', '10y']].shift(1)

# Show the result
date2y10y

,date,2y,10y,source_date
0,2026-09-03,NaN,NaN,2026-09-03
1,2026-09-04,4.34,4.77,2026-09-04
2,2026-09-07,4.37,4.78,2026-09-04
3,2026-09-08,4.37,4.78,2026-09-08
4,2026-09-09,4.39,4.80,2026-09-09
5,2026-09-10,4.43,4.83,2026-09-10
6,2026-09-11,4.56,4.95,2026-09-11
7,2026-09-14,4.63,4.96,2026-09-14
8,2026-09-15,4.65,4.97,2026-09-15
9,2026-09-16,4.67,5.00,2026-09-16


In [12]:
# Download ECB AAA yield curve data
url = (
    "https://data-api.ecb.europa.eu/service/data/"
    "YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_2Y+SR_10Y"
)

response = requests.get(
    url,
    params={
        "startPeriod": start.strftime("%Y-%m-%d"),
        "endPeriod": today.strftime("%Y-%m-%d"),
        "format": "csvdata"
    },
    timeout=60
)
response.raise_for_status()

df_euro = pd.read_csv(StringIO(response.text))

# Keep the two maturities and reshape to columns
df_euro = (
    df_euro[df_euro["DATA_TYPE_FM"].isin(["SR_2Y", "SR_10Y"])]
    .pivot(index="TIME_PERIOD", columns="DATA_TYPE_FM", values="OBS_VALUE")
    .rename(columns={"SR_2Y": "2Y", "SR_10Y": "10Y"})
)

df_euro.columns.name = None
df_euro = df_euro[["2Y", "10Y"]]

df_euro.index = pd.to_datetime(df_euro.index)
df_euro = df_euro.apply(pd.to_numeric, errors="coerce").sort_index()

# Create business-day calendar and carry forward ECB holidays
df_euro = df_euro.reindex(
    pd.date_range(start, today, freq="B")
)
df_euro.index.name = "date"

# Keep track of the actual ECB observation date
df_euro["source_date"] = pd.Series(
    df_euro.index.where(df_euro["2Y"].notna()),
    index=df_euro.index
).ffill()

df_euro[["2Y", "10Y"]] = df_euro[["2Y", "10Y"]].ffill()

# Apply the same one-row lag as the US data
df_euro[["2Y", "10Y", "source_date"]] = (df_euro[["2Y", "10Y", "source_date"]].shift(1))

df_euro = df_euro.reset_index()

df_euro = df_euro[["date", "2Y", "10Y", "source_date"]]

df_euro

,date,2Y,10Y,source_date
0,2026-09-03,NaN,NaN,NaT
1,2026-09-04,2.885664,3.364624,2026-09-03
2,2026-09-07,2.876637,3.346497,2026-09-04
3,2026-09-08,2.913307,3.389493,2026-09-07
4,2026-09-09,2.919754,3.378456,2026-09-08
5,2026-09-10,2.976293,3.426566,2026-09-09
6,2026-09-11,3.100596,3.503209,2026-09-10
7,2026-09-14,3.127569,3.522729,2026-09-11
8,2026-09-15,3.227780,3.540940,2026-09-14
9,2026-09-16,3.187804,3.543701,2026-09-15
